# RT-IDS: Real-Time Network Intrusion Detection for IoT
## PyTorch Deep Neural Network with CIC-IoT-2023 Dataset

This notebook implements Phases 1 & 2 of the thesis:
1. **Phase 1** — Data engineering: CSV → Parquet conversion with Polars
2. **Phase 2** — Deep learning: PyTorch MLP training, evaluation, and persistence

In [ ]:
import os
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
import matplotlib.pyplot as plt

In [ ]:
DATASET_DIRECTORY = 'data/MERGED_CSV/'
PARQUET_PATH = 'data/dataset.parquet'
MODEL_PATH = 'models/ids_dnn.pth'
SCALER_PATH = 'models/scaler.joblib'
ENCODER_PATH = 'models/label_encoder.joblib'

X_COLUMNS = [
    'Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate',
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count',
    'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP',
    'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP',
    'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std',
    'Tot size', 'IAT', 'Number', 'Variance',
]
Y_COLUMN = 'Label'
N_FEATURES = len(X_COLUMNS)

# Binary flag columns (skip log transform)
FLAG_COLUMNS = [
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP',
    'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP',
    'IPv', 'LLC',
]
LOG_COLUMNS = [c for c in X_COLUMNS if c not in FLAG_COLUMNS]

print(f'Feature count: {N_FEATURES}')
print(f'Continuous columns (log1p): {len(LOG_COLUMNS)}')
print(f'Flag columns (untouched): {len(FLAG_COLUMNS)}')

---
## Phase 1: Data Engineering
Convert merged CSVs to a single compressed Parquet file using Polars Lazy API.

In [ ]:
if not os.path.exists(PARQUET_PATH):
    csv_files = sorted([f for f in os.listdir(DATASET_DIRECTORY) if f.endswith('.csv')])
    print(f'Found {len(csv_files)} CSV files')

    lazy_frames = []
    for f in csv_files:
        lf = pl.scan_csv(
            os.path.join(DATASET_DIRECTORY, f),
            infer_schema_length=10000,
        ).select(X_COLUMNS + [Y_COLUMN])
        lazy_frames.append(lf)

    combined = pl.concat(lazy_frames)
    combined.sink_parquet(PARQUET_PATH, compression='zstd')
    print(f'Parquet saved to {PARQUET_PATH}')
else:
    print(f'Parquet already exists at {PARQUET_PATH}')

In [ ]:
df = pl.read_parquet(PARQUET_PATH)
print(f'Dataset shape: {df.shape}')
print(f'\nLabel distribution:')
print(df[Y_COLUMN].value_counts().sort('count', descending=True))

### Data Cleaning Pipeline
1. Drop rows with null labels
2. Replace `inf`/`-inf` with `NaN`, then impute **all** nulls with column median
3. Log1p transformation on continuous columns to compress extreme outlier spikes
4. Sanity check: no nulls, no infs, all values finite float32

In [ ]:
rows_before = df.shape[0]

# 1. Drop rows with null labels
df = df.drop_nulls(subset=[Y_COLUMN])
print(f'Dropped {rows_before - df.shape[0]} rows with null labels')

# 2. Replace inf/-inf with NaN
df = df.with_columns([
    pl.when(pl.col(c).is_infinite())
      .then(None)
      .otherwise(pl.col(c))
      .alias(c)
    for c in X_COLUMNS
])

# 3. Impute ALL nulls (pre-existing + from inf replacement) with column median
null_counts = df.select([pl.col(c).null_count().alias(c) for c in X_COLUMNS])
cols_with_nulls = {c: null_counts[c][0] for c in X_COLUMNS if null_counts[c][0] > 0}
if cols_with_nulls:
    print(f'Columns with nulls (imputed with median):')
    for col, count in sorted(cols_with_nulls.items(), key=lambda x: -x[1]):
        print(f'  {col}: {count}')
    df = df.with_columns([
        pl.col(c).fill_null(pl.col(c).median()).alias(c)
        for c in cols_with_nulls
    ])
else:
    print('No null values found in features')

# 4. Log1p transformation on continuous columns
df = df.with_columns([
    pl.col(c).abs().log1p().alias(c)
    for c in LOG_COLUMNS
])
print(f'\nApplied log1p to {len(LOG_COLUMNS)} continuous columns')

# 5. Final sanity check
remaining_nulls = df.select([pl.col(c).is_null().sum().alias(c) for c in X_COLUMNS]).sum_horizontal()[0]
remaining_infs = df.select([pl.col(c).is_infinite().sum().alias(c) for c in X_COLUMNS]).sum_horizontal()[0]
assert remaining_nulls == 0, f'{remaining_nulls} nulls remain!'
assert remaining_infs == 0, f'{remaining_infs} infs remain!'
print(f'\nClean dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')

---
## Phase 2: Deep Learning Preparation

### Label Mappings
Three classification modes: 34-class (granular), 8-class (attack family), and binary.

In [ ]:
DICT_8CLASSES = {
    'DDOS-RSTFINFLOOD': 'DDoS', 'DDOS-PSHACK_FLOOD': 'DDoS',
    'DDOS-SYN_FLOOD': 'DDoS', 'DDOS-UDP_FLOOD': 'DDoS',
    'DDOS-TCP_FLOOD': 'DDoS', 'DDOS-ICMP_FLOOD': 'DDoS',
    'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS', 'DDOS-ACK_FRAGMENTATION': 'DDoS',
    'DDOS-UDP_FRAGMENTATION': 'DDoS', 'DDOS-ICMP_FRAGMENTATION': 'DDoS',
    'DDOS-SLOWLORIS': 'DDoS', 'DDOS-HTTP_FLOOD': 'DDoS',

    'DOS-UDP_FLOOD': 'DoS', 'DOS-SYN_FLOOD': 'DoS',
    'DOS-TCP_FLOOD': 'DoS', 'DOS-HTTP_FLOOD': 'DoS',

    'MIRAI-GREETH_FLOOD': 'Mirai', 'MIRAI-GREIP_FLOOD': 'Mirai',
    'MIRAI-UDPPLAIN': 'Mirai',

    'RECON-PINGSWEEP': 'Recon', 'RECON-OSSCAN': 'Recon',
    'RECON-PORTSCAN': 'Recon', 'VULNERABILITYSCAN': 'Recon',
    'RECON-HOSTDISCOVERY': 'Recon',

    'DNS_SPOOFING': 'Spoofing', 'MITM-ARPSPOOFING': 'Spoofing',

    'BENIGN': 'Benign',

    'BROWSERHIJACKING': 'Web', 'BACKDOOR_MALWARE': 'Web',
    'XSS': 'Web', 'UPLOADING_ATTACK': 'Web',
    'SQLINJECTION': 'Web', 'COMMANDINJECTION': 'Web',

    'DICTIONARYBRUTEFORCE': 'BruteForce',
}

DICT_2CLASSES = {k: ('Benign' if k == 'BENIGN' else 'Attack') for k in DICT_8CLASSES}

In [ ]:
# Choose classification mode: '34', '8', or '2'
CLASSIFICATION_MODE = '34'

if CLASSIFICATION_MODE == '8':
    df = df.with_columns(pl.col(Y_COLUMN).replace(DICT_8CLASSES).alias(Y_COLUMN))
elif CLASSIFICATION_MODE == '2':
    df = df.with_columns(pl.col(Y_COLUMN).replace(DICT_2CLASSES).alias(Y_COLUMN))

labels = df[Y_COLUMN].unique().sort().to_list()
n_classes = len(labels)
print(f'Classification mode: {CLASSIFICATION_MODE} ({n_classes} classes)')
print(f'Classes: {labels}')

### Train/Test Split & Scaling

In [ ]:
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(df[Y_COLUMN].to_numpy())
X_all = df.select(X_COLUMNS).to_numpy().astype(np.float32)

# 80/20 split
split_idx = int(len(X_all) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')

del df, X_all, y_all

In [ ]:
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

### PyTorch Dataset & DataLoader

In [ ]:
class IDSDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 4096

train_loader = DataLoader(IDSDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(IDSDataset(X_test, y_test), batch_size=BATCH_SIZE)

### MLP Architecture

In [ ]:
class IDSModel(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)


device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = IDSModel(N_FEATURES, n_classes).to(device)
print(model)
print(f'Device: {device}')

### Training Loop with Early Stopping

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_EPOCHS = 50
PATIENCE = 5

best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

for epoch in range(N_EPOCHS):
    # Training
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)
    train_loss = epoch_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    val_loss /= len(test_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f'Epoch {epoch+1:02d}/{N_EPOCHS} — train_loss: {train_loss:.4f} — val_loss: {val_loss:.4f}')

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

model.load_state_dict(best_state)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

### Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(y_batch)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(f'Accuracy:  {accuracy_score(all_labels, all_preds):.4f}')
print(f'Precision: {precision_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'Recall:    {recall_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'F1:        {f1_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'\n{classification_report(all_labels, all_preds, target_names=label_encoder.classes_, zero_division=0)}')

### Save Model, Scaler & Encoder

In [ ]:
os.makedirs('models', exist_ok=True)

torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(label_encoder, ENCODER_PATH)

print(f'Model saved to {MODEL_PATH}')
print(f'Scaler saved to {SCALER_PATH}')
print(f'Label encoder saved to {ENCODER_PATH}')

### Inference Benchmark
Measure single-flow inference time for real-time feasibility.

In [ ]:
import time

model.eval()
sample = X_test[0].unsqueeze(0).to(device)

# Warmup
for _ in range(100):
    with torch.no_grad():
        model(sample)

# Benchmark
n_runs = 1000
start = time.perf_counter()
for _ in range(n_runs):
    with torch.no_grad():
        model(sample)
elapsed = time.perf_counter() - start

avg_us = (elapsed / n_runs) * 1e6
print(f'Single-flow inference: {avg_us:.1f} \u00b5s ({n_runs} runs)')